# Data & AI Job Market — Salary Prediction
## Notebook 3 — Can We Predict a Salary From the Posting?

Author: Francisco Martinez Grecco
Project: [Data-ai-jobs-analysis](https://github.com/FranciscoMartinezGrecco/Data-ai-jobs-analysis)
Previous notebooks: `01_eda.ipynb`, `02_business_insights.ipynb`

## Purpose

The first two notebooks described the market: what salaries look like and which factors move them. This one asks the natural next question: given the attributes of a job posting (role, experience level, country, company size, remote setup), how well can we actually predict the salary?

A couple of caveats before starting. The goal here is the modeling workflow itself (feature preparation, honest baselines, validation, interpretation), not squeezing out the last decimal of accuracy. And with only 565 postings and fairly coarse features, the expectations should stay modest: a big part of what sets an individual salary, like specific skills, the company, or how someone negotiated, simply isn't in this dataset. Seeing how far those visible attributes actually get us is part of the point.

## Approach

1. Move the target to log scale to tame the right tail.
2. Select features, avoiding leakage and near-constant columns.
3. Build a preprocessing + model pipeline with scikit-learn.
4. Compare a median baseline, linear regression, random forest and gradient boosting.
5. Interpret the best model with permutation importance and error analysis.

---

## Section 1 — Setup and Data

Same loading and cleaning pipeline as the previous notebooks, plus the modeling helpers from `src/model.py`.

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.inspection import permutation_importance

# Add the parent folder to the path so we can import from src/
import sys
sys.path.append('..')

from src.clean import load_raw_data, clean_salaries_df
from src.features import build_features
from src.model import (collapse_rare_categories, build_preprocessor,
                       build_model_pipelines, regression_metrics)

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Same style as the other notebooks
sns.set_theme(style='whitegrid', palette='viridis')

# One seed for everything, so the notebook gives the same numbers on every run
RANDOM_STATE = 42

print("Imports done.")

Imports done.


In [2]:
# Load and prepare the data with the same pipeline as Notebooks 1 and 2
df_raw = load_raw_data('../data/raw/ds_salaries.csv')
df = clean_salaries_df(df_raw)
df = build_features(df)

print(f"\nDataset ready: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Median salary: ${df['salary_in_usd'].median():,.0f}")

Loaded 607 rows × 12 columns
Removed 42 duplicate rows
Cleaned dataset: 565 rows × 11 columns
Features added. New shape: 565 rows × 14 columns

Dataset ready: 565 rows, 14 columns
Median salary: $100,000


## Section 2 — The Target in Log Scale

Salaries are heavily right-skewed, something Notebook 1 already showed in detail (`salary_log_transform.png`). Training on raw dollar amounts would let a handful of $400k+ salaries dominate the fit, so the models work with `log1p(salary_in_usd)` and predictions are converted back to dollars for reporting. The log transform actually overshoots a little and leaves a mild left tail, but it is far closer to symmetric than the raw values.

In [3]:
y_usd = df['salary_in_usd']
y_log = np.log1p(y_usd)

print(f"Skewness of salary_in_usd:  {y_usd.skew():.2f}")
print(f"Skewness after log1p:      {y_log.skew():.2f}")

Skewness of salary_in_usd:  1.73
Skewness after log1p:      -1.20


## Section 3 — Feature Selection and Engineering

The dataset offers eight potential predictors, but not all of them make the cut, and the two geographic columns need work first: `job_title` and `company_location` both have 50 unique values, far too many to one-hot encode with only 565 rows.

In [4]:
# job_title and company_location have 50 categories each. One-hot
# encoding them as-is would create a sparse matrix full of columns the
# model sees three or four times. We keep the frequent categories
# (top 10 titles cover ~80% of postings) and group the rest as Other.
model_df = df.copy()
model_df['job_title_grp'] = collapse_rare_categories(model_df['job_title'], top_n=10)
model_df['location_grp'] = collapse_rare_categories(model_df['company_location'], top_n=8)

# Whether the employee lives in a different country than the company.
# Same flag used for the cross-border analysis in Notebook 2.
model_df['cross_border'] = (model_df['employee_residence'] != model_df['company_location']).astype(int)

print(f"job_title: 50 categories -> {model_df['job_title_grp'].nunique()}")
print(f"company_location: 50 categories -> {model_df['location_grp'].nunique()}")
print(f"cross-border postings: {model_df['cross_border'].sum()}")

job_title: 50 categories -> 11
company_location: 50 categories -> 9
cross-border postings: 51


In [5]:
# Final feature set. A few deliberate exclusions:
# - salary and salary_currency would leak the target directly
# - employment_type is 97% Full-Time, so it carries almost no signal
# - employee_residence is redundant once we have location + cross_border
# - remote_ratio is just the numeric version of remote_category
nominal_features = ['job_title_grp', 'location_grp', 'remote_category']
numeric_features = ['experience_rank', 'company_size_rank', 'work_year', 'cross_border']

X = model_df[nominal_features + numeric_features]

X_train, X_test, ylog_train, ylog_test, yusd_train, yusd_test = train_test_split(
    X, y_log, y_usd, test_size=0.2, random_state=RANDOM_STATE)

print(f"Features: {list(X.columns)}")
print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

Features: ['job_title_grp', 'location_grp', 'remote_category', 'experience_rank', 'company_size_rank', 'work_year', 'cross_border']
Train: 452 rows | Test: 113 rows


## Section 4 — Preprocessing Pipeline

Everything goes through a single `ColumnTransformer` wrapped in a scikit-learn `Pipeline` per model. This is more than tidiness: because preprocessing lives inside the pipeline, it gets refit on the training portion of every cross-validation fold, so no information from the validation folds leaks into the encoders or the scaler.

In [6]:
# One-hot for the nominal columns, standard scaling for the numeric
# ones. Scaling is irrelevant for the trees but helps the linear model,
# and sharing one preprocessor keeps all pipelines identical.
preprocessor = build_preprocessor(nominal_features, numeric_features)
models = build_model_pipelines(preprocessor, random_state=RANDOM_STATE)

print("Models to compare:")
for name in models:
    print(f"  - {name}")

Models to compare:
  - Baseline (median)
  - Linear Regression
  - Random Forest
  - Gradient Boosting
